# 05: Agrupar datos: GROUP BY y HAVING

Agrupar filas por categoría y filtrar grupos: la herramienta que convierte datos en informes.



## [01] Clientes por ciudad

El director comercial quiere saber cuántos clientes tenemos en cada ciudad y cuántos son VIP. 

1. Agrupa la tabla clientes por ciudad y muestra: 
    - ciudad, total de clientes, 
2. clientes VIP (COUNT con CASE)
3. porcentaje de VIPs, de 0 a 100 y con un decimal. 
4. Ordena por el porcentaje de VIPs, de mayor a menor:
    - es el ORDER BY que ya trae el código de partida.



**Resultado esperado**

| ciudad     | total_clientes | clientes_vip | pct_vip |
| ---------- | -------------- | ------------ | ------- |
| Teruel     | 8              | 2            | 25      |
| Soria      | 5              | 1            | 20      |
| Murcia     | 92             | 12           | 13      |
| Cádiz      | 40             | 5            | 12.5    |
| Málaga     | 139            | 17           | 12.2    |
| Bilbao     | 277            | 33           | 11.9    |
| Madrid     | 1755           | 206          | 11.7    |
| Valencia   | 600            | 70           | 11.7    |
| Barcelona  | 1016           | 117          | 11.5    |
| NULL       | 375            | 43           | 11.5    |
| Zaragoza   | 184            | 21           | 11.4    |
| Sevilla    | 416            | 47           | 11.3    |
| Valladolid | 60             | 5            | 8.3     |
| Santander  | 12             | 1            | 8.3     |
| Girona     | 25             | 2            | 8       |

## Solución propia (con FILTER)


```sql
-- Clientes por ciudad con tasa de VIPs
SELECT
    ciudad,
    COUNT(*) AS total_clientes, 
    COUNT(*) FILTER (WHERE es_vip) AS clientes_vip,
    ROUND(100.0 * COUNT(*) FILTER (WHERE es_vip) / COUNT(*), 1) AS pct_vip
FROM clientes
GROUP BY ciudad
ORDER BY pct_vip DESC; 
```

Solución Legacy (CASE WHEN)


```sql
-- Clientes por ciudad con tasa de VIPs
SELECT
    ciudad,
    COUNT(*) AS total_clientes, 
    COUNT(CASE WHEN es_vip THEN 1 END) AS clientes_vip,
    ROUND(100.0 * COUNT(CASE WHEN es_vip THEN 1 END) / COUNT(*), 1) AS pct_vip
FROM clientes
GROUP BY ciudad
ORDER BY pct_vip DESC; 
```

## [02] Top 10 clientes por gasto

Finanzas quiere identificar los 10 clientes más valiosos.

1. Agrupa los pedidos completados por cliente_id 
2. Muestra, para cada cliente, cuántos pedidos tiene (num_pedidos)
3. Gasto total redondeado a dos decimales (gasto_total).
4.  Solo los 10 primeros.



| cliente_id | num_pedidos | gasto_total |
| ---------- | ----------- | ----------- |
| 2840       | 25          | 3729.50     |
| 2670       | 27          | 3680.17     |
| 820        | 24          | 3465.06     |
| 2561       | 27          | 3423.92     |
| 2743       | 28          | 3379.15     |
| 711        | 26          | 3296.88     |
| 541        | 27          | 3268.57     |
| 2719       | 27          | 3253.10     |
| 2731       | 26          | 3228.55     |
| 529        | 27          | 3199.15     |

```sql
-- Top 10 clientes por gasto total
SELECT
    cliente_id,
    COUNT(*) AS num_pedidos,
    ROUND(SUM(importe), 2) AS gasto_total
FROM pedidos
WHERE estado = 'completado'
GROUP BY ciente_id
ORDER BY gasto_total DESC NULLS LAST, cliente_id
LIMIT 10;
```

## [03] Clientes valiosos con HAVING

El programa de fidelización necesita clientes que hayan gastado más de 500€ en total (solo completados). 

1. Muestra cliente_id, número de pedidos y gasto total. 
2. Usa HAVING para filtrar. 
3. El ORDER BY y el LIMIT 10 del final ya están puestos: 
    - la comprobación automática solo mira las 10 primeras filas, y sin un orden fijo no siempre son las mismas. 
4. Fíjate en que el orden es ASCENDENTE: 
    - te salen los que JUSTO pasan de 500€, que es donde se ve si tu HAVING está haciendo algo.



```sql
-- Clientes con gasto > 500€ (HAVING)
SELECT
    cliente_id,
    COUNT(*) AS pedidos,
    ROUND(SUM(importe), 2) AS gasto_total
FROM pedidos
WHERE estado = 'completado'
GROUP BY cliente_id
HAVING gasto_total > 500
ORDER BY gasto_total, cliente_id
LIMIT 10;
```

## [04] Evolución mensual de ventas

El CFO quiere ver la evolución mensual de 2024. 

1. Para cada mes, muestra: 
    - número de pedidos completados 
    - total facturado.
2. Usa DATE_TRUNC para agrupar por mes: 
    - DATE_TRUNC('month', fecha) convierte cada fecha en el día 1 de su mes
    - así que todos los pedidos de marzo caen en el mismo grupo. 
3. Es todo lo que necesitas saber de la función para este ejercicio; 
    - las funciones de fecha se explican con calma en «Fechas sin bugs». 
4. El ORDER BY y el LIMIT 10 del final ya están puestos: 
    - la comprobación automática solo mira las 10 primeras filas
    - sin un orden fijo no siempre son las mismas.



**Resultado esperado**

| mes        | pedidos | total    |
| ---------- | ------- | -------- |
| 2024-01-01 | 896     | 91705.10 |
| 2024-02-01 | 870     | 90401.24 |
| 2024-03-01 | 897     | 93561.55 |
| 2024-04-01 | 785     | 84418.55 |
| 2024-05-01 | 675     | 69544.39 |
| 2024-06-01 | 667     | 68029.44 |
| 2024-07-01 | 528     | 53284.85 |
| 2024-08-01 | 489     | 49518.27 |
| 2024-09-01 | 356     | 37745.26 |
| 2024-10-01 | 311     | 34208.47 |

```sql
-- Ventas mensuales de 2024
SELECT
    DATE_TRUNC('month', fecha) AS mes,
    COUNT(*) AS pedidos,
    ROUND(SUN(importe), 2) AS total
FROM pedidos
WHERE estado = 'completado'
    AND fecha >= '2024-01-01'
    AND fecha < '2025-01-01'
GROUP BY DATE_TRUNC('month', fecha)
ORDER BY mes
LIMIT 10;
```

1. DATE_TRUNC('month', fecha) "trunca" la fecha al primer día del mes. Así, 2024-03-15, 2024-03-22 y 2024-03-01 todas se convierten en 2024-03-01 → se agrupan juntas.

2. Es la función estándar para agregar datos temporales por período (día, semana, mes, trimestre, año).

3. Importante: en GROUP BY repetimos la expresión completa DATE_TRUNC('month', fecha), no el alias "mes". Algunos motores (DuckDB, PostgreSQL) aceptan el alias en GROUP BY, pero no todos — repetir la expresión es más portable.

4. ORDER BY mes ordena cronológicamente (enero, febrero, marzo...) porque las fechas se ordenan naturalmente.

5. Este patrón (GROUP BY DATE_TRUNC + ORDER BY cronológico) es el patrón #1 para gráficos de series temporales en dashboards.

## [05] Rendimiento por canal de adquisición

**Marketing quiere saber qué canal de adquisición trae más clientes.**

1. Agrupa por canal_adquisicion y muestra: 
    - canal
    - total de clientes
    - clientes VIP
    - la tasa de VIPs.
2.  Solo canales con más de 800 clientes (HAVING).



**Resultado esperado**

| canal_adquisicion | total_clientes | clientes_vip | tasa_vip |
| ----------------- | -------------- | ------------ | -------- |
| email             | 890            | 106          | 11.9     |
| google            | 889            | 103          | 11.6     |
| referido          | 889            | 100          | 11.2     |
| organico          | 890            | 100          | 11.2     |
| facebook          | 889            | 99           | 11.1     |

```sql
-- Rendimiento por canl de adquisición
SELECT
    canal_adquisicion,
    COUNT(*) AS total_clientes,
    COUNT(*) FILTER (WHERE es_vip) AS clientes_vip,
    ROUND(100.0 * COUNT(*) FILTER (WHERE es_vip) / COUNT(*), 1) AS tasa_vip
FROM cliente
GROUP BY canal_adquisicion
HAVING COUNT(*) > 800
ORDER BY tasa_vip DESC;
```

Combina todos los conceptos de la lección: GROUP BY + COUNT CASE + HAVING.

- GROUP BY canal_adquisicion → un grupo por canal (email, redes sociales, orgánico, etc.).
- COUNT(CASE WHEN es_vip THEN 1 END) → dentro de cada canal, cuenta solo los VIP.
- HAVING COUNT(*) > 800 → filtra canales pequeños (menores de 800 clientes no son relevantes estadísticamente).

Este reporte responde a una pregunta clave de marketing: "¿qué canal atrae clientes de mayor valor?". Un canal puede traer muchos clientes pero pocos VIP, y viceversa.

Nota: HAVING usa COUNT(*) (la expresión completa), no el alias total_clientes. Aunque DuckDB lo permite, es mejor práctica usar la expresión para portabilidad.

## [06] La ciudad donde mejor funcionan los referidos, con un mínimo de clientes

La directora de marketing quiere premiar a la ciudad donde mejor funciona el programa de referidos el de «trae a un amigo», y está convencida de que es Teruel: 
- alguien le ha enseñado un ranking en el que sale primera con un 25 %. 

Antes de que mande el premio, rehaz el ranking con un mínimo: 
1. solo cuentan las ciudades con al menos 50 clientes, que es lo que el equipo ha acordado para comparar ciudades. 
2. Para cada ciudad:
    - muestra los clientes, 
    - cuántos llegaron por referido (canal_adquisicion = referido) 
    - el porcentaje. 

El SELECT, el WHERE y el ORDER BY con su LIMIT 5 ya están puestos y no hay que tocarlos: te falta el mínimo.

**Resultado esperado**

| ciudad     | clientes | referidos | pct_referidos |
| ---------- | -------- | --------- | ------------- |
| Murcia     | 92       | 18        | 19.6          |
| Bilbao     | 277      | 51        | 18.4          |
| Valladolid | 60       | 11        | 18.3          |
| Málaga     | 139      | 25        | 18            |
| Barcelona  | 1016     | 182       | 17.9          |

```sql
-- Ranking de ciudades por porcentaje de clientes que llegaron por referido
SELECT
    ciudad,
    COUNT(*) AS clientes,
    COUNT(*) FILTER (WHERE canal_adquisicion = 'referido') AS referidos,
    ROUND(100.0 * COUNT(*) FILTER (WHERE canal_adquisicion = 'referido') / COUNT(*), 1) AS pct_referidos
FROM clientes
WHERE ciudad IS NOT FULL
GROUPY BY ciudad
HAVING COUNT(*) >= 50
ORDER BY pct_referidos DESC, ciudad
LIMIT 5;

```

Una línea, HAVING COUNT(*) >= 50, y el ranking cambia de arriba abajo.

Sin ella, los tres primeros son Teruel (2 referidos de 8 clientes, un 25 %), Girona (5 de 25) y Soria (1 de 5). Con ella desaparecen los tres, y la primera es Murcia, con un 19,6 %. Pero mira la distancia: entre Murcia y Barcelona, la quinta, hay 1,7 puntos, y Murcia tiene 92 clientes, así que un par de referidos más o menos la cambiarían de puesto. La respuesta para la directora de marketing no es «el premio para Murcia»: es «Teruel salía primera porque tiene 8 clientes, y en las ciudades con suficientes clientes el programa funciona parecido en todas, en torno al 18 %».

Por qué HAVING y no WHERE: la condición es sobre el tamaño del grupo, COUNT(*), y ese número no existe hasta que se ha agrupado. WHERE trabaja con filas sueltas y no puede saber cuántas hay en cada ciudad.

Por qué >= y no >: la regla dice «al menos 50», así que una ciudad con 50 justos entra. Aquí no cambia el resultado, pero en un informe de verdad esa frontera es la que alguien te va a preguntar.

Error común: poner el mínimo en el porcentaje (HAVING pct_referidos > ...). Lo que hay que exigir no es un porcentaje alto, sino un grupo con casos suficientes para que su porcentaje signifique algo.

